# Testing non-linear cyclpoint-related activation in fMRI

In [48]:
%matplotlib inline
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nibabel as nib
from nilearn.maskers import NiftiLabelsMasker
import statsmodels.api as sm
from statsmodels.gam.api import GLMGam, CyclicCubicSplines
import multiprocessing as mp
import warnings
warnings.filterwarnings('ignore') 

## Read in fMRI contrast and cyclepoint data

In [49]:
df = pd.read_csv('parcel_activation.csv')
nparcels = 100 # number of parcels

## Define GAM cyclepoint function

In [50]:
def run_gam(i):
    cc = CyclicCubicSplines(df['cyclepoint'],df=[10])
    stat = sm.gam.GLMGam.from_formula(f'parcel_{i}~cyclepoint',data=df,smoother=cc).fit().test_significance(0)
    return([stat.pvalue,stat.statistic[0][0]])

## Run GAM analysis

In [51]:
results = pd.DataFrame(columns=['parcel', 'pvalue', 'statistic'])
for i in range(nparcels):
    res = run_gam(i+1)
    results = pd.concat([results,
                         pd.DataFrame({'parcel': [i+1], 'pvalue': res[0], 'statistic': [res[1]]})],
                         ignore_index=True)
results.to_csv('cyclepoint_activation_results.csv', index=False)

In [52]:
atlas = NiftiLabelsMasker('Schaefer2018_100Parcels_7Networks_order_FSLMNI152_2mm.nii.gz').fit()
results_chisq = atlas.inverse_transform(results['statistic'].values)
results_p = atlas.inverse_transform(-np.log10(results['pvalue']))
nib.save(results_chisq, 'cyclepoint_activation_chisquare.nii.gz')
nib.save(results_p, 'cyclepoint_activation_neglogp.nii.gz')